In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from rasterstats import point_query, zonal_stats
from scipy import stats
from scipy.ndimage import uniform_filter, maximum_filter, minimum_filter
from shapely.geometry import Point
import os, glob, warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
NLCD_PATH   = r'C:\sp\Annual_NLCD_LndCov_2024_CU_C1V1_c37a1cb6-bb15-49cb-afa4-5f037356b6d3.tiff'
DEP_PATH    = r'C:\sp\OilGasLocations_ConventionalUnconventional2025_12.shp'
DERIV_FOLDER = r'C:\users\colto\documents\github\lidar_project\data\derivatives'
FULL_DEM     = r'C:\users\colto\documents\github\lidar_project\data\full_dem.vrt'

RASTERS = {
    'tpi_5m':    os.path.join(DERIV_FOLDER, 'tpi_5m.tif'),
    'tpi_15m':   os.path.join(DERIV_FOLDER, 'tpi_15m.tif'),
    'tpi_50m':   os.path.join(DERIV_FOLDER, 'tpi_50m.tif'),
    'slope':     os.path.join(DERIV_FOLDER, 'slope.tif'),
    'curvature': os.path.join(DERIV_FOLDER, 'curvature.tif'),
    'relief':    os.path.join(DERIV_FOLDER, 'relief_10m.tif'),
    'roughness': os.path.join(DERIV_FOLDER, 'roughness.tif'),
}

# NLCD class code → readable label
NLCD_CLASSES = {
    11: 'Open Water', 21: 'Developed Open', 22: 'Developed Low',
    23: 'Developed Med', 24: 'Developed High', 31: 'Barren',
    41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
    52: 'Shrub/Scrub', 71: 'Grassland', 81: 'Pasture/Hay',
    82: 'Cultivated Crops', 90: 'Woody Wetlands', 95: 'Emergent Wetlands'
}

print("Setup complete.")

Setup complete.


In [2]:
# ── Load DEP current wells — source of spud dates ────────────────────────────
print("Loading DEP current wells...")
dep = gpd.read_file(DEP_PATH)
dep = dep.to_crs('EPSG:26917')

# Parse SPUD_DATE, replace 1800-01-01 placeholder with NaT
dep['SPUD_DATE'] = pd.to_datetime(dep['SPUD_DATE'], errors='coerce')
dep.loc[dep['SPUD_DATE'].dt.year < 1850, 'SPUD_DATE'] = pd.NaT
dep['spud_year'] = dep['SPUD_DATE'].dt.year
print(f"DEP wells with valid spud date: {dep['spud_year'].notna().sum():,}")

# ── Clip DEP wells to LiDAR extent ───────────────────────────────────────────
with rasterio.open(FULL_DEM) as src:
    dem_bounds = src.bounds

from shapely.geometry import box
dem_box  = box(dem_bounds.left, dem_bounds.bottom, dem_bounds.right, dem_bounds.top)
dem_gdf  = gpd.GeoDataFrame(geometry=[dem_box], crs='EPSG:26917')
dep_clip = gpd.clip(dep, dem_gdf).reset_index(drop=True)
print(f"DEP wells inside LiDAR extent: {len(dep_clip)}")

# ── Spatial join: nearest DEP well to each historic well ─────────────────────
# For each PADEP/USGS well, find the closest DEP well within 150m
# and inherit its spud year as an age proxy
print("Joining age from nearest DEP well (within 150m)...")

wells_for_age = wells_clipped.copy().reset_index(drop=True)

# sjoin_nearest finds closest match; max_distance limits to 150m
age_join = gpd.sjoin_nearest(
    wells_for_age[['geometry']],
    dep_clip[['geometry', 'spud_year', 'WELL_TYPE', 'WELL_STATU', 'DATE_PLUGG']],
    how='left',
    max_distance=150,
    distance_col='match_dist_m'
)

# Keep only the first match per well (sjoin_nearest can return duplicates)
age_join = age_join[~age_join.index.duplicated(keep='first')]

wells_for_age['spud_year']    = age_join['spud_year'].values
wells_for_age['match_dist_m'] = age_join['match_dist_m'].values
wells_for_age['dep_status']   = age_join['WELL_STATU'].values
wells_for_age['date_plugged'] = pd.to_datetime(age_join['DATE_PLUGG'].values, errors='coerce')

# Decade bins for grouping
wells_for_age['decade'] = (wells_for_age['spud_year'] // 10 * 10).astype('Int64')

matched = wells_for_age['spud_year'].notna().sum()
print(f"Wells matched to a DEP record: {matched} / {len(wells_for_age)}")
print(f"\nDecade distribution of matched wells:")
print(wells_for_age['decade'].value_counts().sort_index())

Loading DEP current wells...
DEP wells with valid spud date: 115,577


RasterioIOError: C:\users\colto\documents\github\lidar_project\data\full_dem.vrt: No such file or directory

In [ ]:
# ── Extract NLCD land cover at each well ─────────────────────────────────────
print("Extracting NLCD land cover...")

# NLCD is in Albers (EPSG:5070) — reproject wells to match
wells_albers = wells_for_age.to_crs('EPSG:5070')
nlcd_vals    = point_query(wells_albers, NLCD_PATH, interpolate='nearest')
wells_for_age['nlcd_code']  = [int(v) if v is not None else None for v in nlcd_vals]
wells_for_age['nlcd_label'] = wells_for_age['nlcd_code'].map(NLCD_CLASSES).fillna('Unknown')

print(f"\nLand cover distribution at well locations:")
print(wells_for_age['nlcd_label'].value_counts())

# ── Derive terrain classes from existing derivatives ─────────────────────────
print("\nExtracting terrain features at well locations...")

features = {}
for name, path in RASTERS.items():
    features[name] = point_query(wells_for_age, path)

feat_df = pd.DataFrame(features)
wells_for_age = pd.concat([wells_for_age.reset_index(drop=True), feat_df], axis=1)

# Classify slope into terrain context bins
wells_for_age['slope_class'] = pd.cut(
    wells_for_age['slope'],
    bins=[0, 5, 15, 30, 90],
    labels=['Flat (0-5°)', 'Gentle (5-15°)', 'Moderate (15-30°)', 'Steep (>30°)']
)

# Classify local relief into bins
wells_for_age['relief_class'] = pd.cut(
    wells_for_age['relief'],
    bins=[0, 2, 5, 10, 100],
    labels=['Very Low (<2m)', 'Low (2-5m)', 'Moderate (5-10m)', 'High (>10m)']
)

# Drop rows with no terrain features
wells_enriched = wells_for_age.dropna(subset=list(RASTERS.keys())).reset_index(drop=True)
print(f"\nWells with full terrain features: {len(wells_enriched)}")
print(wells_enriched[['type','nlcd_label','slope_class','relief_class','decade']].describe())

In [ ]:
# ── Helper: violin plot by category ──────────────────────────────────────────
def violin_by_category(df, feature, groupby, title, ax, palette=None):
    groups   = df[groupby].dropna().unique()
    groups   = sorted([g for g in groups if pd.notna(g)])
    data     = [df.loc[df[groupby] == g, feature].dropna().values for g in groups]
    data     = [(d, g) for d, g in zip(data, groups) if len(d) > 5]
    if not data:
        ax.set_visible(False)
        return
    vals, labels = zip(*data)
    parts = ax.violinplot(vals, showmedians=True, showextrema=False)
    for i, pc in enumerate(parts['bodies']):
        pc.set_alpha(0.7)
        pc.set_facecolor(plt.cm.cividis(i / max(len(vals)-1, 1)))
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(2)
    ax.set_xticks(range(1, len(labels)+1))
    ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(feature)

# ── Helper: Mann-Whitney U test table ────────────────────────────────────────
def mw_test_table(wells, nonwell_vals, feature_cols):
    rows = []
    for feat in feature_cols:
        w = wells[feat].dropna()
        n = nonwell_vals[feat].dropna() if hasattr(nonwell_vals, '__getitem__') else nonwell_vals
        if len(w) < 5 or len(n) < 5:
            continue
        stat, p = stats.mannwhitneyu(w, n, alternative='two-sided')
        rows.append({
            'feature':    feat,
            'well_median': w.median(),
            'bg_median':  n.median(),
            'p_value':    p,
            'significant': '*' if p < 0.05 else ''
        })
    return pd.DataFrame(rows).sort_values('p_value')

feature_cols = list(RASTERS.keys())

# ── Figure 1: Wells vs background — overall distributions ────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

# Need background samples — reuse nonwell_features from all_samples if available
# otherwise re-extract
bg_df = all_samples[all_samples['label'] == 0][feature_cols]
w_df  = wells_enriched[feature_cols]

for i, feat in enumerate(feature_cols):
    ax = axes[i]
    ax.violinplot([bg_df[feat].dropna(), w_df[feat].dropna()],
                  showmedians=True, showextrema=False)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Background', 'Well'], fontsize=10)
    ax.set_title(feat, fontsize=10)
    # Add p-value annotation
    stat, p = stats.mannwhitneyu(w_df[feat].dropna(), bg_df[feat].dropna(),
                                  alternative='two-sided')
    ax.set_xlabel(f'p={p:.4f}{"*" if p < 0.05 else ""}', fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Terrain Feature Distributions: Wells vs Background', fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\char_overall.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: By well type ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, feat in enumerate(feature_cols):
    violin_by_category(wells_enriched, feat, 'type',
                       f'{feat} by well type', axes[i])
axes[-1].set_visible(False)
plt.suptitle('Terrain Features by Well Type', fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\char_by_type.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: By land cover ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, feat in enumerate(feature_cols):
    violin_by_category(wells_enriched, feat, 'nlcd_label',
                       f'{feat} by land cover', axes[i])
axes[-1].set_visible(False)
plt.suptitle('Terrain Features by Land Cover', fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\char_by_landcover.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 4: By terrain slope context ───────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, feat in enumerate(feature_cols):
    violin_by_category(wells_enriched, feat, 'slope_class',
                       f'{feat} by slope class', axes[i])
axes[-1].set_visible(False)
plt.suptitle('Terrain Features by Slope Context', fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\char_by_slope.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 5: By decade drilled ───────────────────────────────────────────────
age_wells = wells_enriched[wells_enriched['decade'].notna()].copy()
age_wells['decade'] = age_wells['decade'].astype(str)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
for i, feat in enumerate(feature_cols):
    violin_by_category(age_wells, feat, 'decade',
                       f'{feat} by decade drilled', axes[i])
axes[-1].set_visible(False)
plt.suptitle('Terrain Features by Decade Drilled', fontsize=13)
plt.tight_layout()
plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\char_by_decade.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Statistical test summary ──────────────────────────────────────────────────
print("\n=== Mann-Whitney U: Wells vs Background ===")
print("Null hypothesis: well and background distributions are identical")
print("* = statistically significant at p < 0.05\n")
mw = mw_test_table(w_df, bg_df, feature_cols)
print(mw.to_string(index=False))